In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2026-08-01
Revised on 2026-08-02

@author:       Oscar Trevizo, Visiting Professor
@institution:  DeVry University
@context:      SIS150 -- Fundamentals of Programming
@environment:  Python 3.14.3 | myenv | MacBook Air M5

Module 5: Object-Oriented Programming -- Inheritance
======================================================

Description:
    Live, in-class walkthrough notebook for SIS150 Module 5. Short,
    cell-by-cell demo run on the instructor's screen while students
    follow along and reproduce each cell themselves. Parallels zyBooks
    CEIS150 "Programming with Objects" (Module 2, Chapter 5) using
    original, non-duplicated examples. Continues the music theme from
    Module 4, but this notebook is self-contained -- every class it
    needs is redefined here from scratch, so it runs standalone without
    ever opening the Module 4 notebook.

    A three-level inheritance chain drives most of the notebook:
        Song            -- the abstract composition: title, composer,
                           where/when it was composed (a PartialDateTime
                           -- year required, month/day optional)
        SongRecording   -- one specific recorded performance (IS-A Song):
                           artist, duration, engineer, studio, location,
                           and a recording date/time (also a
                           PartialDateTime), plus take_number and tape_id
        ReleasedTake    -- the specific take chosen for official release
                           (IS-A SongRecording): adds a release date
                           (also a PartialDateTime) and overrides play()
                           to note it's the released version

    PartialDateTime is a small has-a composition class: it holds
    year/month/day/hour/minute (all but year optional -- None means
    unknown), defines __str__ so it prints only the precision that's
    actually known, and provides as_datetime() to convert to a real
    datetime.datetime when one is needed (filling any unknown
    month/day/hour/minute with 1/1/0/0). Song, SongRecording, and
    ReleasedTake each hold a PartialDateTime as an attribute rather than
    loose year/month/day fields.

    Topics covered:
        Derived classes -- inheriting from a base class, calling the
            base class constructor explicitly (parallels zyBooks 5.1)
        Multi-level inheritance -- attribute/method lookup climbing the
            three-level chain above (5.2)
        Overriding methods -- replacing a base method vs. extending it
            with an explicit call to the base version (5.3)
        Is-a vs. has-a -- inheritance (the chain above) contrasted with
            three composition examples: Playlist has-a a growing list of
            SongRecordings, Vinyl45 has-a exactly two (an optional
            B-side, addable after construction) plus the commercial
            attributes (purchase price/year, current value) that
            deliberately don't belong on the is-a chain, and
            PartialDateTime itself (Song/SongRecording/ReleasedTake each
            has-a one). Vinyl45 shown correct first, then BuggyVinyl45
            demonstrates the common mistake of calling a base class's
            __init__(self, ...) when self isn't that type, versus
            calling the class itself to build a genuinely separate
            object (5.4)
        The unittest module -- a couple of real unit tests against the
            classes built above (5.5)
        Two mini-challenges paralleling the Module 5 labs:
            - FamilyPlan derived from StreamingPlan, overriding cost
              (parallels the Instrument info lab, 5.6)
            - SpecialtyStoreInventory derived from RecordStoreInventory,
              extending report() (parallels the Pet info lab, 5.7)

Revision History:
    2026-08-01  Original development -- live-coding demo for Module 5
    2026-08-01  Renamed VinylRecord -> VinylPressing; added Vinyl45 (a
                has-a composition example) after recognizing a physical
                single realistically holds two songs, not one
    2026-08-01  Moved commercial attributes (purchase price/year,
                current value) off the is-a chain entirely, onto Vinyl45
    2026-08-01  Major restructuring: inserted SongRecording between Song
                and VinylPressing, since duration/artist need to vary
                per recording (a live take differs from a studio cut;
                the same song can have a different performing artist)
    2026-08-02  Second major restructuring: (1) Song gained
                date_composed/place_composed metadata with partial-
                precision support. (2) SongRecording's recording_date
                got the same treatment. (3) VinylPressing removed --
                "pressing" stopped making sense once the chain was about
                recordings and releases, not vinyl manufacturing.
                (4) Added ReleasedTake(SongRecording) in its place,
                restoring the three-level chain and giving the
                overriding-methods demo a home. (5) Reordered the
                Vinyl45 has-a section to show the correct class first,
                BuggyVinyl45 afterward as a cautionary example.
    2026-08-02  Third pass: promoted partial-date handling from a
                function (format_partial_date) plus a plain dict on each
                class, into a proper PartialDateTime class -- Song,
                SongRecording, and ReleasedTake each now has-a
                PartialDateTime instead of a dict. Added as_datetime()
                to convert to a real datetime.datetime when one is
                needed, filling unknown month/day/hour/minute with
                1/1/0/0 -- a deliberate, documented tradeoff (this loses
                the ability to tell "really January 1st" from "unknown,
                defaulted to January 1st"; __str__ still shows only the
                genuinely known precision). Removed the now-redundant
                *_display() methods in favor of PartialDateTime's own
                __str__.
"""


'\nCreated on 2026-08-01\nRevised on 2026-08-02\n\n@author:       Oscar Trevizo, Visiting Professor\n@institution:  DeVry University\n@context:      SIS150 -- Fundamentals of Programming\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nModule 5: Object-Oriented Programming -- Inheritance\n======================================================\n\nDescription:\n    Live, in-class walkthrough notebook for SIS150 Module 5. Short,\n    cell-by-cell demo run on the instructor\'s screen while students\n    follow along and reproduce each cell themselves. Parallels zyBooks\n    CEIS150 "Programming with Objects" (Module 2, Chapter 5) using\n    original, non-duplicated examples. Continues the music theme from\n    Module 4, but this notebook is self-contained -- every class it\n    needs is redefined here from scratch, so it runs standalone without\n    ever opening the Module 4 notebook.\n\n    A three-level inheritance chain drives most of the notebook:\n        Song            -- t

# SIS150 Module 5 - Object-Oriented Programming: Inheritance

Author: Prof. Oscar Trevizo

Date: August 1, 2026

## Reference

https://docs.python.org/3/tutorial/classes.html#inheritance

## Derived classes

A class can be built *on top of* another class instead of from scratch.
The new class -- the **derived class** -- automatically gets everything
the **base class** has, plus whatever new attributes and methods it
adds. `Song` here is deliberately thin: it's the abstract composition,
true no matter who ever performs it.

One more thing before we start: real historical dates are often only
*partly* known -- we might know the year a song was composed, but not
the month or day. `PartialDateTime` is a small has-a class built for
exactly that: it holds whatever precision is known, prints only that
much, and can convert to a real `datetime.datetime` on request.

In [2]:
from datetime import datetime

class PartialDateTime:
    """A date/time that might only be partially known."""
    def __init__(self, year, month=None, day=None, hour=None, minute=None):
        self.year = year
        self.month = month     # None if unknown
        self.day = day          # None if unknown
        self.hour = hour         # None if unknown
        self.minute = minute      # None if unknown

    def __str__(self):
        text = f"{self.year}"
        for value in (self.month, self.day):
            if value is None:
                break
            text += f"-{value:02d}"
        if self.hour is not None:
            text += f" {self.hour:02d}"
            if self.minute is not None:
                text += f":{self.minute:02d}"
        return text

    def as_datetime(self):
        """Convert to a real datetime.datetime. Fills any unknown
        month/day/hour/minute with 1/1/0/0 -- once converted, "really
        January 1st" and "unknown, defaulted to January 1st" look
        identical, so prefer str(self) when you just need to display
        the date."""
        return datetime(self.year, self.month or 1, self.day or 1,
                         self.hour or 0, self.minute or 0)

In [3]:
for sample in (PartialDateTime(1969), PartialDateTime(1968, 7),
               PartialDateTime(1968, 7, 31), PartialDateTime(1968, 7, 31, 14, 30)):
    print(f"{sample}  -->  as_datetime(): {sample.as_datetime()}")

1969  -->  as_datetime(): 1969-01-01 00:00:00
1968-07  -->  as_datetime(): 1968-07-01 00:00:00
1968-07-31  -->  as_datetime(): 1968-07-31 00:00:00
1968-07-31 14:30  -->  as_datetime(): 1968-07-31 14:30:00


In [4]:
class Song:
    """A song -- the composition itself, independent of any performance."""
    def __init__(self, title, composer, place_composed, year_composed,
                 month_composed=None, day_composed=None):
        self.title = title
        self.composer = composer
        self.place_composed = place_composed
        self.date_composed = PartialDateTime(year_composed, month_composed, day_composed)

In [5]:
# Only the year is known for when this one was composed
song1 = Song("Come Together", "Lennon-McCartney",
             place_composed="England", year_composed=1969)

print(song1.title, "-- composed by", song1.composer)
print(f"Composed: {song1.date_composed}, in {song1.place_composed}")

Come Together -- composed by Lennon-McCartney
Composed: 1969, in England


A specific recorded performance of that song **is a** `Song` -- it has
a title and a composer -- plus a lot that's only true of *this
recording*, not the song in the abstract: who performed it, how long it
runs, and exactly where/when/how it was recorded. Duration and artist
don't belong on `Song` itself, because they change from recording to
recording -- a live take runs longer than a studio cut, and the same
song can be performed by a different artist entirely.
`SongRecording` derives from `Song`, calling `Song`'s own constructor
explicitly to set up the parts it inherits. Its own recording date/time
is also a `PartialDateTime`.

In [6]:
class SongRecording(Song):   # SongRecording derives from (IS-A) Song
    """One specific recorded performance of a song."""
    def __init__(self, title, composer, place_composed, year_composed,
                 artist, duration_sec, engineer, studio, location,
                 recording_year, take_number, tape_id,
                 month_composed=None, day_composed=None,
                 recording_month=None, recording_day=None,
                 recording_hour=None, recording_minute=None):
        Song.__init__(self, title, composer, place_composed, year_composed,
                      month_composed, day_composed)   # set up the Song part
        self.artist = artist
        self.duration_sec = duration_sec
        self.engineer = engineer
        self.studio = studio
        self.location = location
        self.recording_date = PartialDateTime(recording_year, recording_month,
                                               recording_day, recording_hour,
                                               recording_minute)
        self.take_number = take_number
        self.tape_id = tape_id

    def play(self):
        print(f"Now playing: {self.title} by {self.artist}")

    def duration_display(self):
        minutes = self.duration_sec // 60
        seconds = self.duration_sec % 60
        return f"{minutes}:{seconds:02d}"

In [7]:
# Illustrative take/tape details -- not asserted as the real session log
hey_jude_take = SongRecording(
    title="Hey Jude", composer="Lennon-McCartney",
    place_composed="Surrey, England", year_composed=1968,
    artist="The Beatles", duration_sec=431, engineer="Barry Sheffield",
    studio="Trident Studios", location="London, England",
    recording_year=1968, recording_month=7, recording_day=31,
    take_number=8, tape_id="Reel 4")

print(hey_jude_take.title, "--", hey_jude_take.artist)      # title: inherited; artist: own
print("Composer:", hey_jude_take.composer)                    # composer: inherited from Song too
print(hey_jude_take.duration_display())
print(f"Composed: {hey_jude_take.date_composed}")    # found in Song -- inherited
print(f"Recorded: {hey_jude_take.recording_date}")    # own attribute

Hey Jude -- The Beatles
Composer: Lennon-McCartney
7:11
Composed: 1968
Recorded: 1968-07-31


## Multi-level inheritance

A derived class can itself be a base class for another derived class --
that's a three-level chain. `ReleasedTake` derives from `SongRecording`,
which derives from `Song`. An instance three levels down still has
access to everything at every level above it. (A song is often recorded
many times -- different takes, different sessions -- but only one take
usually becomes *the* released version. That's what `ReleasedTake`
marks.)

In [8]:
class ReleasedTake(SongRecording):   # derives from SongRecording, which derives from Song
    """The specific take chosen for official release, among possibly many takes recorded."""
    def __init__(self, title, composer, place_composed, year_composed,
                 artist, duration_sec, engineer, studio, location,
                 recording_year, take_number, tape_id, release_year,
                 month_composed=None, day_composed=None,
                 recording_month=None, recording_day=None,
                 recording_hour=None, recording_minute=None,
                 release_month=None, release_day=None):
        SongRecording.__init__(self, title, composer, place_composed, year_composed,
                                artist, duration_sec, engineer, studio, location,
                                recording_year, take_number, tape_id,
                                month_composed, day_composed,
                                recording_month, recording_day,
                                recording_hour, recording_minute)
        self.release_date = PartialDateTime(release_year, release_month, release_day)

In [9]:
released1 = ReleasedTake(
    title="Revolution", composer="Lennon-McCartney",
    place_composed="England", year_composed=1968,
    artist="The Beatles", duration_sec=203, engineer="Barry Sheffield",
    studio="Abbey Road Studios", location="London, England",
    recording_year=1968, recording_month=7, recording_day=10,
    take_number=18, tape_id="Reel 11",
    release_year=1968, release_month=8, release_day=26)

print(released1.title)                          # found in Song           -- two levels up
print(released1.duration_display())              # found in SongRecording -- one level up
print(f"Recorded: {released1.recording_date}")    # found in SongRecording -- one level up
print(f"Released: {released1.release_date}")      # found in ReleasedTake -- its own class

Revolution
3:23
Recorded: 1968-07-10
Released: 1968-08-26


## Overriding methods

A derived class can define a method with the **same name** as one in
its base class -- that *overrides* the base version. Playing the
officially released take should still count as a play (so it makes
sense to reuse `SongRecording.play()`), but it's worth noting that this
particular take is *the* released one. Calling the base method
explicitly and then adding more is *extending* the base behavior, not
replacing it.

In [10]:
class ReleasedTake(SongRecording):
    """The specific take chosen for official release -- now with its own play() behavior."""
    def __init__(self, title, composer, place_composed, year_composed,
                 artist, duration_sec, engineer, studio, location,
                 recording_year, take_number, tape_id, release_year,
                 month_composed=None, day_composed=None,
                 recording_month=None, recording_day=None,
                 recording_hour=None, recording_minute=None,
                 release_month=None, release_day=None):
        SongRecording.__init__(self, title, composer, place_composed, year_composed,
                                artist, duration_sec, engineer, studio, location,
                                recording_year, take_number, tape_id,
                                month_composed, day_composed,
                                recording_month, recording_day,
                                recording_hour, recording_minute)
        self.release_date = PartialDateTime(release_year, release_month, release_day)

    def play(self):
        SongRecording.play(self)                     # extend, don't replace, the base version
        print(f"  (the officially released take -- released {self.release_date})")

In [11]:
# A fresh ReleasedTake instance, built from the redefined class above
record1 = ReleasedTake(
    title="Revolution", composer="Lennon-McCartney",
    place_composed="England", year_composed=1968,
    artist="The Beatles", duration_sec=203, engineer="Barry Sheffield",
    studio="Abbey Road Studios", location="London, England",
    recording_year=1968, recording_month=7, recording_day=10,
    take_number=18, tape_id="Reel 11",
    release_year=1968, release_month=8, release_day=26)

print("Playing a plain SongRecording:")
hey_jude_take.play()

print("\nPlaying a ReleasedTake:")
record1.play()

Playing a plain SongRecording:
Now playing: Hey Jude by The Beatles

Playing a ReleasedTake:
Now playing: Revolution by The Beatles
  (the officially released take -- released 1968-08-26)


## Is-a versus has-a

`ReleasedTake` **is a** `SongRecording`, which **is a** `Song` -- three
levels of inheritance, already built above. A `Playlist` is a different
relationship: it isn't a kind of recording, it just *contains* some. A
`Playlist` **has a** list of `SongRecording` objects -- that's
*composition*, not inheritance. (`PartialDateTime` is also a has-a
relationship, by the way -- every `Song` has-a `PartialDateTime` for
when it was composed, not is-a one.)

In [12]:
class Playlist:
    """A playlist HAS-A collection of recordings -- composition, not inheritance."""
    def __init__(self, playlist_name):
        self.playlist_name = playlist_name
        self.songs = []   # a Playlist HAS-A list of SongRecording objects

    def add_song(self, song):
        self.songs.append(song)

    def total_duration(self):
        return sum(s.duration_sec for s in self.songs)

    def show(self):
        print(f"Playlist: {self.playlist_name}")
        for s in self.songs:
            print(f"  {s.title} by {s.artist} ({s.duration_display()})")

In [13]:
road_trip = Playlist("Road Trip Mix")
road_trip.add_song(hey_jude_take)   # a SongRecording
road_trip.add_song(record1)          # a ReleasedTake -- fits right in, since it IS-A SongRecording too
road_trip.show()
print(f"Total duration: {road_trip.total_duration()} seconds")

Playlist: Road Trip Mix
  Hey Jude by The Beatles (7:11)
  Revolution by The Beatles (3:23)
Total duration: 634 seconds


### A second has-a example: a 45 RPM single

A real 45 RPM single almost always has **two** songs -- an A-side and a
B-side. That rules out inheritance entirely: a disc with two recordings
on it doesn't hold up as "is-a SongRecording." It's composition, same
idea as `Playlist`, just holding exactly two slots instead of a growing
list -- and this is also where the *commercial* side lives: what
someone actually paid for the physical disc, and what it's worth now.

In [14]:
class Vinyl45:
    """A 45 RPM single -- HAS-A recording on side A, and optionally one on side B."""
    def __init__(self, song_a, purchase_price, purchase_year, song_b=None):
        self.song_a = song_a   # a whole, separate SongRecording object
        self.song_b = song_b   # optional -- some singles start without a B-side assigned
        self.purchase_price = purchase_price   # price paid for the physical disc itself
        self.purchase_year = purchase_year

    def current_value(self, current_year, annual_growth=0.08):
        age = current_year - self.purchase_year
        return self.purchase_price * (1 + annual_growth) ** age

    def show(self):
        print(f"Side A: {self.song_a.title} by {self.song_a.artist}")
        if self.song_b is not None:
            print(f"Side B: {self.song_b.title} by {self.song_b.artist}")
        else:
            print("Side B: (not yet assigned)")

In [15]:
# The actual 1968 single pairing -- "Hey Jude" b/w "Revolution"
single1 = Vinyl45(hey_jude_take, purchase_price=4.00, purchase_year=1968)   # side B left blank
single1.show()

print()
single1.song_b = record1   # add the B-side after the fact -- a ReleasedTake fits fine, it IS-A SongRecording
single1.show()
print(f"Worth ${single1.current_value(2026):,.2f} in 2026")

Side A: Hey Jude by The Beatles
Side B: (not yet assigned)

Side A: Hey Jude by The Beatles
Side B: Revolution by The Beatles
Worth $347.25 in 2026


It's tempting, the first time you build something like `Vinyl45`, to try
reusing a base class's constructor the same way `SongRecording` and
`ReleasedTake` do above. Here's why that specific move doesn't work
once a class *isn't* inheriting from that base:

In [16]:
class BuggyVinyl45:
    """Doesn't work -- see the output below, then compare to the working Vinyl45 above."""
    def __init__(self, title_a, composer_a, place_composed_a, year_composed_a):
        # BUG: BuggyVinyl45 does NOT derive from Song (it's not an
        # "is-a Song" -- look, no "(Song)" after the class name), so
        # calling Song.__init__(self, ...) doesn't build a separate Song
        # object. It runs Song's setup code directly on self instead.
        self.song_a = Song.__init__(self, title_a, composer_a, place_composed_a, year_composed_a)

buggy = BuggyVinyl45("Hey Jude", "Lennon-McCartney", "Surrey, England", 1968)
print("buggy.song_a is:", buggy.song_a)                 # not a Song object at all!
print("buggy.title exists?", hasattr(buggy, "title"), "-- value:", buggy.title)

buggy.song_a is: None
buggy.title exists? True -- value: Hey Jude


Two things went wrong at once: `song_a` ended up `None`, because
`__init__` never *returns* anything -- it only sets up `self` in place.
And `title` landed directly on the `BuggyVinyl45` instance instead,
because `Song.__init__(self, ...)` runs Song's setup code using
*whatever `self` you hand it*, and here that was the single itself, not
a separate Song.

Compare that to the working `Vinyl45` above: it never calls
`Song.__init__` at all -- it just receives an already-built
`SongRecording` object (`hey_jude_take`, `record1`) and stores it. The
fix, if you did need to build one from scratch inline, is to call the
class itself -- `Song(...)`, not `Song.__init__(self, ...)` -- which
actually *builds and returns* a brand-new, independent object.

## Testing your code: the `unittest` module

Writing tests that check a method's output against a known, correct
answer is called **unit testing**. Python's built-in `unittest` module
handles running the tests and reporting which passed or failed.

In [17]:
import unittest

class TestMusicClasses(unittest.TestCase):
    def test_songrecording_duration_display(self):
        rec = SongRecording("Test Track", "Test Composer", "Test City", 2020,
                             "Test Artist", 125, "Test Engineer", "Test Studio",
                             "Test City", 2020, 1, "Reel 1")
        self.assertEqual(rec.duration_display(), "2:05")

    def test_partial_date_time_str(self):
        pdt = PartialDateTime(1969)
        self.assertEqual(str(pdt), "1969")

    def test_vinyl45_gains_value(self):
        rec = SongRecording("Test Track", "Test Composer", "Test City", 2020,
                             "Test Artist", 200, "Test Engineer", "Test Studio",
                             "Test City", 2020, 1, "Reel 1")
        single = Vinyl45(rec, purchase_price=10.00, purchase_year=2016)
        self.assertGreater(single.current_value(2026), single.purchase_price)

runner = unittest.TextTestRunner(verbosity=2)
runner.run(unittest.TestLoader().loadTestsFromTestCase(TestMusicClasses))

test_partial_date_time_str (__main__.TestMusicClasses.test_partial_date_time_str) ... ok
test_songrecording_duration_display (__main__.TestMusicClasses.test_songrecording_duration_display) ... ok
test_vinyl45_gains_value (__main__.TestMusicClasses.test_vinyl45_gains_value) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.001s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## Mini-challenge: a FamilyPlan derived from StreamingPlan

Parallels the zyBooks *Instrument info* lab -- a derived class that adds
its own attribute and *extends* (not replaces) the base class's
calculation.

In [18]:
class StreamingPlan:
    """A monthly music streaming plan."""
    def __init__(self, plan_name="Free Tier", hours_streamed=0.0):
        self.plan_name = plan_name
        self.hours_streamed = hours_streamed

    def monthly_cost(self):
        if self.plan_name == "Free Tier":
            return 0.00
        return self.hours_streamed * 0.10

In [19]:
class FamilyPlan(StreamingPlan):
    """A shared plan for multiple listeners."""
    def __init__(self, hours_streamed, num_members):
        StreamingPlan.__init__(self, plan_name="Family", hours_streamed=hours_streamed)
        self.num_members = num_members   # FamilyPlan's own attribute

    def monthly_cost(self):
        base_cost = StreamingPlan.monthly_cost(self)   # extend the base calculation
        return base_cost + (self.num_members * 2.00)    # $2/member on top

In [20]:
family = FamilyPlan(hours_streamed=60.0, num_members=4)
print(f"{family.plan_name}: ${family.monthly_cost():.2f}/month "
      f"for {family.num_members} members")

Family: $14.00/month for 4 members


## Mini-challenge: a SpecialtyStoreInventory derived from RecordStoreInventory

Parallels the zyBooks *Pet info* lab -- a derived class that adds one
attribute and extends the base class's `report()` method.

In [21]:
class RecordStoreInventory:
    """Tracks vinyl record stock for one title at a record store."""
    def __init__(self, starting_stock=50):
        self.stock = starting_stock

    def sell(self, count):
        self.stock -= count

    def restock(self, count):
        self.stock += count

    def report(self):
        print(f"Inventory: {self.stock} records")

In [22]:
class SpecialtyStoreInventory(RecordStoreInventory):
    """A record store inventory focused on one genre."""
    def __init__(self, starting_stock, genre_focus):
        RecordStoreInventory.__init__(self, starting_stock)
        self.genre_focus = genre_focus   # SpecialtyStoreInventory's own attribute

    def report(self):
        RecordStoreInventory.report(self)   # extend the base report
        print(f"  Genre focus: {self.genre_focus}")

In [23]:
shop = SpecialtyStoreInventory(starting_stock=20, genre_focus="Punk")
shop.sell(5)
shop.restock(2)
shop.report()

Inventory: 17 records
  Genre focus: Punk


## Let's review

- A **derived class** inherits everything from its **base class**, and
  adds its own attributes/methods -- an *is-a* relationship. This
  notebook builds one three levels deep: `ReleasedTake` is-a
  `SongRecording` is-a `Song`.
- The base class constructor must be called **explicitly** on `self`
  (e.g. `Song.__init__(self, ...)`) for a derived class to set up the
  inherited attributes -- but only when `self` genuinely *is* the base
  type. If it isn't, call the class itself instead (`Song(...)`) to get
  back a whole separate object to store as an attribute.
- **Multi-level inheritance**: attribute and method lookup climbs the
  whole chain -- `ReleasedTake` -> `SongRecording` -> `Song`.
- A derived class can **override** a base method entirely, or **extend**
  it by explicitly calling the base version first, then adding more.
- **Composition** (*has-a*, like `Playlist` holding a list of
  `SongRecording` objects, `Vinyl45` holding exactly two, or a `Song`
  holding a `PartialDateTime`) is a different relationship from
  **inheritance** (*is-a*).
- **Where an attribute belongs matters**: `Song` holds only what's true
  of the composition itself (title, composer, when/where composed);
  `SongRecording` holds what's true of one specific performance
  (artist, duration, session details); `ReleasedTake` marks which one
  became official; `Vinyl45` holds the commercial facts (price, value)
  that belong to the physical object someone actually owns.
- **Partial precision, promoted to a class**: `PartialDateTime` bundles
  year/month/day/hour/minute (all but year optional) with its own
  `__str__` (shows only what's known) and `as_datetime()` (converts to
  a real `datetime.datetime`, filling unknown pieces with 1/1/0/0 -- a
  deliberate, documented tradeoff, not a bug).
- The **`unittest`** module runs repeatable tests against your code and
  reports pass/fail.

Next: Module 6 covers AI-assisted coding with Python libraries.